In [1]:
# --- Cell 2: Imports ---
from rdflib import Graph

# Notice we use 'src.filename' now!
from schema_definition import build_schema
from integrate_rQTLs import add_rqtl_to_graph
from integrate_gwas_catalog import enrich_graph_with_gwas
from integrate_open_targets import enrich_graph_with_opentargets
#from integrate_reactome import enrich_graph_with_reactome
from schema_definition import *
from execute_sparql import execute_sparql
from graphdb_engine import query_graphdb

In [19]:
# 1. Initialize Graph
import random
from tqdm import tqdm
import glob
g = Graph()
g = build_schema(g)
number_of_rqtls= 10
rqtl_files = glob.glob("../data/json_files/*.json")
rqtl_files= random.sample(rqtl_files, number_of_rqtls)
# 2. Define Schema & Ontologies
rqtl_json_path = "../data/json_files/GCST90200330_GCST90200020.json"

# 3. Ingest Local JSON Data
print("\n[STEP 1] Ingesting local rQTL data...")
#add_rqtl_to_graph(rqtl_json_path, g)
for rqtl_file in tqdm(rqtl_files,total=number_of_rqtls):
    g = add_rqtl_to_graph(rqtl_file, g)





[STEP 1] Ingesting local rQTL data...


100%|██████████| 10/10 [00:00<00:00, 15.88it/s]


In [20]:
import numpy as np
# 4. Enrich with External APIs
number_of_snps= len(list(g.subjects(RDF.type, MTR.SNP)))

print("\n[STEP 2] Enriching with External Databases...")
g = enrich_graph_with_gwas(g,max_snps=number_of_snps)



[STEP 2] Enriching with External Databases...

--- Starting GWAS Catalog Integration (v2 API) ---
[1/12] Fetching v2 associations for rs1047891...
   -> Found 10 significant associations!
[2/12] Fetching v2 associations for rs6687264...
   -> Found 1 significant associations!
[3/12] Fetching v2 associations for rs77420750...
   -> Found 7 significant associations!
[4/12] Fetching v2 associations for rs270615...
   -> Found 7 significant associations!
[5/12] Fetching v2 associations for rs603424...
   -> Found 10 significant associations!
[6/12] Fetching v2 associations for rs3812316...
   -> Found 10 significant associations!
[7/12] Fetching v2 associations for rs4665972...
   -> Found 10 significant associations!
[8/12] Fetching v2 associations for rs968567...
   -> Found 10 significant associations!
[9/12] Fetching v2 associations for rs438568...
   -> Found 10 significant associations!
[10/12] Fetching v2 associations for rs1177442...
   -> Found 10 significant associations!
[11/12

In [21]:
execute_sparql(g,"../analysis/query_gwas_catalog.rq")

Loaded query from file: ../analysis/query_gwas_catalog.rq
Executing SPARQL query...
Success! Query returned 0 rows.


,snp_id,disease_name,gwas_p_value,ratio_name,causal_gene


In [22]:
number_of_genes=len(list(g.subjects(RDF.type, MTR.Gene)))


In [23]:

g = enrich_graph_with_opentargets(g, max_genes=number_of_genes)



--- Starting Open Targets Integration ---
Found 48 causal Genes in the local graph.
[1/48] Fetching OT data for CYP4Z1...
   -> Found 10 top associated diseases!
[2/48] Fetching OT data for CYP4A22...
   -> Found 1 safety liabilities!
   -> Found 10 top associated diseases!
[3/48] Fetching OT data for CYP4A11...
   -> Found 1 safety liabilities!
   -> Found 10 top associated diseases!
[4/48] Fetching OT data for TAL1...
   -> Found 10 top associated diseases!
[5/48] Fetching OT data for AC004791.2...
   -> AC004791.2 not found in Open Targets database.
[6/48] Fetching OT data for CYP4F11...
   -> Found 10 top associated diseases!
[7/48] Fetching OT data for SLC22A5...
   -> Found 1 safety liabilities!
   -> Found 10 top associated diseases!
[8/48] Fetching OT data for AC034220.3...
   -> AC034220.3 not found in Open Targets database.
[9/48] Fetching OT data for LYRM7...
   -> Found 10 top associated diseases!
[10/48] Fetching OT data for SLC22A4...
   -> Found 10 top associated diseas

In [34]:
from graphdb_engine import query_graphdb

# The optimized query from earlier
optimized_gwas_query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX oban: <http://purl.org/oban/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?snp ?ratio_name ?trait_name ?gwas_p_value ?gwas_beta
WHERE {
  # 1. Get the rQTL Associations
  ?ratio a mtr:MetaboliteRatio ;
         rdfs:label ?ratio_name .

  ?rqtl_assoc a oban:association ;
              oban:has_object ?ratio ;
              oban:has_subject ?snp .

  # 2. Get the GWAS Associations for those exact SNPs
  ?gwas_assoc a biolink:Association ;
              biolink:has_subject ?snp ;
              biolink:has_object ?trait ;
              mtr:p_value ?gwas_p_value .

  ?trait rdfs:label ?trait_name .

  # 3. Grab the beta if it exists
  OPTIONAL { ?gwas_assoc mtr:beta ?gwas_beta . }

  # 4. Filter for genome-wide significance
  FILTER (?gwas_p_value < 0.00000005)
}
ORDER BY ?gwas_p_value
LIMIT 10

"""

# Execute the query against GraphDB!
# Note: Ensure the 'repo_name' matches what you named it in the GraphDB Workbench
df_results = query_graphdb(optimized_gwas_query, repo_name="mtrKG")

# Display the results
display(df_results)

Sending query to GraphDB (mtrKG)...
Success! Retrieved 10 rows.


,snp,ratio_name,trait_name,gwas_p_value,gwas_beta
0,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.35285
1,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.352894
2,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.352233
3,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.351561
4,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.350063
5,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.348978
6,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.245665
7,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,5.0000000000000003e-284,-0.35285
8,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,5.0000000000000003e-284,-0.352894
9,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,5.0000000000000003e-284,-0.352233


In [25]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX oban: <http://purl.org/oban/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?snp ?ratio_name ?trait_name ?gwas_p_value ?gwas_beta
WHERE {
  ?rqtl_assoc a oban:association ;
              oban:has_subject ?snp ;
              oban:has_object ?ratio .
  ?ratio rdfs:label ?ratio_name .

  ?gwas_assoc a biolink:Association ;
              biolink:has_subject ?snp ;
              biolink:has_object ?trait ;
              mtr:p_value ?gwas_p_value .
  ?trait rdfs:label ?trait_name .

  OPTIONAL { ?gwas_assoc mtr:beta ?gwas_beta . }

  FILTER (?gwas_p_value < 0.00000005)
}
ORDER BY ?gwas_p_value
LIMIT 10
"""

In [15]:
print(g.serialize(format="turtle"))

@prefix biolink: <https://w3id.org/biolink/vocab/> .
@prefix chebi: <http://purl.obolibrary.org/obo/CHEBI_> .
@prefix chembl: <http://identifiers.org/chembl.compound/> .
@prefix dbsnp: <http://identifiers.org/dbsnp/> .
@prefix efo: <http://www.ebi.ac.uk/efo/> .
@prefix hmdb: <http://identifiers.org/hmdb/> .
@prefix mtr: <https://metabolite-ratio-app.unil.ch/rqtl/> .
@prefix ns1: <http://purl.org/oban/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix pubmed: <https://pubmed.ncbi.nlm.nih.gov/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sio: <http://semanticscience.org/resource/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

chembl:CHEMBL1431 a biolink:Drug ;
    rdfs:label "METFORMIN" ;
    prov:wasDerivedFrom <https://platform.opentargets.org/> ;
    mtr:max_clinical_phase "4"^^xsd:float ;
    biolink:targets <http://identifiers.org/hgnc.symbol/NDU

In [33]:
df_results = query_graphdb(optimized_gwas_query, repo_name="mtrKG")

# Display the results
display(df_results)

Sending query to GraphDB (mtrKG)...
Success! Retrieved 0 rows.


,snp_id,disease_name,gwas_p_value,ratio_name,causal_gene


In [30]:
execute_sparql(g,optimized_gwas_query)

Executing SPARQL query...


KeyboardInterrupt: 

In [37]:
execute_sparql(g,optimized_gwas_query)

Executing SPARQL query...


KeyboardInterrupt: 

In [31]:
g.serialize(destination="../output/mtrKG.ttl", format="turtle")

<Graph identifier=N8cb74d146b8c4127b84bf7b9a841e155 (<class 'rdflib.graph.Graph'>)>

In [ ]:
g = enrich_graph_with_reactome(g)

# 5. NetworkX Graph Analytics
print("\n[STEP 3] Running Graph Analytics...")
g = calculate_reaction_distances(g)

# 6. SHACL Validation
print("\n[STEP 4] Validating Graph structure...")
validate_graph_with_shacl(g)

# 7. Save Output
output_path = "output/final_knowledge_graph.ttl"
g.serialize(destination=output_path, format="turtle")
print(f"\n[SUCCESS] Knowledge Graph saved to {output_path} with {len(g)} triples!")